<a href="https://colab.research.google.com/github/Yahimeen/IA_AplicadaLLama_Challenge_02_01/blob/main/Hands_On_WhatsApp_Cloud_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: WHATSAPP CLOUD API**

Una vez vista la masterclass ***WhatsApp Cloud API***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Ollama** para correr Llama 3.1 de forma local dentro de Colab, y **ngrok** para exponer el servidor mientras se desarrolla.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/19F84m5PKCMKEOHxL7z3MUqN98HEhwLWS?usp=drive_link).

## **LOGICA DE NEGOCIO Y MODELO**

### **CONFIGURACIÓN DEL ENTORNO**

#### **COLAB SECRETS**

Para no exponer tu API key directamente en el código, Colab ofrece un panel de Secrets (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la API key para guardarla dentro de una variable y usarla dentro del notebook.

In [17]:
# Leer credenciales desde Colab Secrets

from google.colab import userdata

VERIFY_TOKEN = "mi_token"  # debe coincidir con el configurado en el dashboard de Meta
WHATSAPP_TOKEN = userdata.get('WHATSAPP_ACCESS_TOKEN')
PHONE_NUMBER_ID = userdata.get('PHONE_NUMBER_ID')  # visible en API Setup del dashboard de Meta
#PHONE_NUMBER_ID = '1287616061108559'
PHONE_TEST_NUMBER = userdata.get('WHATSAPP_TEST_NUMBER')  # visible en API Setup del dashboard de Meta
NGROK_AUTHTOKEN = userdata.get('NGROK_AUTHTOKEN')  # visible en API Setup del dashboard de Meta
print("Credenciales cargadas correctamente.")


MessageError: TypeError: Failed to fetch

#### **INSTALACIÓN Y ARRANQUE DE OLLAMA**

Ollama no viene preinstalado en la máquina virtual de Google Colab, por lo que se debe instalar y arrancar en segundo plano al inicio de cada sesión.

In [2]:
# Instalar Ollama
!apt-get update -qq
!apt-get install -y zstd # requiere zstd, que Colab no trae por defecto
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 51 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (1,119 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/lo

In [3]:
# Iniciar el servidor de Ollama
import subprocess
import time

proceso_ollama = subprocess.Popen(["ollama", "serve"])
time.sleep(5)  # dar tiempo a que el servidor levante

!pip install ollama --quiet
import ollama

!ollama pull llama3.1:8b
print("Ollama listo.")


Ollama listo.


### **CONSTRUCCIÓN DE LA LÓGICA DEL ASISTENTE**

Con Ollama corriendo, se arma la lógica del asistente: la estructura del servidor con sus registros en memoria, la normalización de números mexicanos (necesaria para que la API de envío acepte el número que entrega el webhook), y la función que genera la respuesta real con Llama 3.1.

In [4]:
# Configurar el servidor del webhook
from fastapi import FastAPI, Request, BackgroundTasks
import requests
import ollama

app = FastAPI()

mensajes_recibidos = []
usuarios_conocidos = set()  # numeros que ya recibieron el saludo inicial

In [5]:
# Definir función de normalización de número
def normalizar_numero_mx(numero):
    #if numero.startswith("521") and len(numero) == 13:
    #    return "52" + numero[3:]
    return numero

In [6]:
# Definir función de generación de respuesta
def generar_respuesta(texto_usuario):
    prompt = (
        "Eres el asistente de atención a clientes de un gimnasio. "
        "Responde de forma breve (2-3 líneas) a la siguiente pregunta sobre horarios o membresías:\n\n"
        + texto_usuario
    )

    respuesta = ollama.chat(model="llama3.1:8b", messages=[{"role": "user", "content": prompt}])
    return respuesta["message"]["content"]

## **DESPLIEGUE DEL SERVIDOR**

### **DEFINIR LOS ENDPOINTS DEL WEBHOOK**

Se definen los dos endpoints que WhatsApp necesita: uno para la verificación inicial del webhook, y otro para recibir cada mensaje entrante.

La recepción delega el trabajo pesado a `BackgroundTasks`, para responderle a Meta de inmediato sin esperar a que Llama genere la respuesta, evitando así que Meta reintente la entrega y se dupliquen las respuestas.

In [7]:
# Definir endpoint de verificación de credenciales

@app.get("/webhook")
async def verificar_webhook(request: Request):
    params = request.query_params

    if params.get("hub.verify_token") == VERIFY_TOKEN:
        return int(params.get("hub.challenge"))

    return {"error": "token invalido"}


In [8]:
# Definir endpoint de recepción de mensajes
@app.post("/webhook")
async def recibir_mensaje(request: Request, background_tasks: BackgroundTasks):
    body = await request.json()
    mensajes_recibidos.append(body)

    try:
        mensaje = body["entry"][0]["changes"][0]["value"]["messages"][0]
        numero = normalizar_numero_mx(mensaje["from"])
        #numero = mensaje["from"]

        texto = mensaje["text"]["body"]
        background_tasks.add_task(procesar_y_responder, numero, texto)
    except (KeyError, IndexError):
        pass

    return {"status": "ok"}  # Meta recibe el ACK de inmediato, sin esperar a Ollama

In [9]:
# Definir función de envío de respuesta
def enviar_respuesta(numero, texto):
    print(numero)
    url = f"https://graph.facebook.com/v25.0/{PHONE_NUMBER_ID}/messages"
    headers = {"Authorization": f"Bearer {WHATSAPP_TOKEN}", "Content-Type": "application/json"}
    payload = {
        "messaging_product": "whatsapp",
        "to": numero,
        "type": "text",
        "text": {"body": texto}
    }

    r = requests.post(url, headers=headers, json=payload)
    print("Envio a", numero, "->", r.status_code, r.text)

In [10]:
# Definir función de procesamiento de respuesta
respuestas_enviadas = []

def enviar_respuesta(numero, texto):
    url = f"https://graph.facebook.com/v25.0/{PHONE_NUMBER_ID}/messages"
    headers = {"Authorization": f"Bearer {WHATSAPP_TOKEN}", "Content-Type": "application/json"}
    payload = {
        "messaging_product": "whatsapp",
        "to": numero,
        "type": "text",
        "text": {"body": texto}
    }

    r = requests.post(url, headers=headers, json=payload)
    respuestas_enviadas.append({"numero": numero, "status": r.status_code, "texto": r.text})

In [11]:
# Definir función de procesamiento de respuesta

def procesar_y_responder(numero, texto):
    if numero not in usuarios_conocidos:
        usuarios_conocidos.add(numero)
        enviar_respuesta(numero, "¡Hola! Soy el asistente del gimnasio. ¿En qué puedo ayudarte?")
    else:
        respuesta = generar_respuesta(texto)
        enviar_respuesta(numero, respuesta)

### **EXPONER EL SERVIDOR CON NGROK**

Se abre un túnel público con ngrok y se levanta el servidor en un hilo aparte, para que Meta pueda entregarle los mensajes reales al webhook. La URL impresa (+ `/webhook`) es la que se configura como *Callback URL* en Meta for Developers.

\begin{equation} WhatsApp  \, \, \, \Rightarrow \, \, \, Configuración  \, \,de  \, \,la  \, \,API  \, \, \, \Rightarrow \, \, \, Configurar  \, \,webhooks \end{equation}

In [15]:
# Instalar y arrancar servidor + túnel público con ngrok
!pip install pyngrok --quiet

from pyngrok import ngrok
import uvicorn
import threading
import time

# Cerrar túneles viejos de esta sesión, si existen
try:
    ngrok.kill()
except Exception:
    pass

# Si el servidor se ejecutó antes en este runtime, pedirle que termine
try:
    servidor.should_exit = True
    time.sleep(1)
except Exception:
    pass

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info",
)
servidor = uvicorn.Server(config)

def correr_servidor():
    servidor.run()

hilo = threading.Thread(target=correr_servidor, daemon=True)
hilo.start()

# Dar tiempo a Uvicorn para abrir el puerto
time.sleep(2)

ngrok.set_auth_token("NGROK_AUTHTOKEN")
tunel_publico = ngrok.connect(8000, "http")

CALLBACK_URL = tunel_publico.public_url + "/webhook"

print("Servidor iniciado.")
print("\nConfigura en Meta:")
print("Callback URL:", CALLBACK_URL)
print("Verify Token: usa el valor del secreto WHATSAPP_VERIFY_TOKEN")
print("\nDespués suscríbete al campo 'messages' del webhook.")


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [1662]
INFO:     Started server process [1662]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


ERROR:pyngrok.process.ngrok:t=2026-09-29T20:06:30+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: NGROK_AUTHTOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"
ERROR:pyngrok.process.ngrok:t=2026-09-29T20:06:30+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: NGROK_AUTHTOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"


PyngrokNgrokError: The ngrok process errored on start: authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: NGROK_AUTHTOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n.

### **PROBAR EL FLUJO COMPLETO**

Se manda un primer mensaje para confirmar que el saludo inicial funciona, y un segundo mensaje para confirmar que ya se recibe una respuesta generada por Llama.

In [ ]:
# Mandar mensaje al bot para recibir un saludo
payload_1 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": "12033002267", "text": {"body": "Hola desde Curso 2"}}
    ]}}]}]
}

r1 = requests.post("http://127.0.0.1:8000/webhook", json=payload_1)
print("Primer mensaje:", r1.status_code, r1.text)


INFO:     127.0.0.1:33514 - "POST /webhook HTTP/1.1" 200 OK
Primer mensaje: 200 {"status":"ok"}


In [ ]:
import requests

url = f"https://graph.facebook.com/v23.0/{PHONE_NUMBER_ID}/messages"

headers = {
    "Authorization": f"Bearer {WHATSAPP_TOKEN}",
    "Content-Type": "application/json"
}

data = {
    "messaging_product": "whatsapp",
    "to": "12033002267",
    "type": "template",
    "template": {
        "name": "hello_world",
        "language": {
            "code": "en_US"
        }
    }
}

response = requests.post(url, headers=headers, json=data)
print(response.json())

{'messaging_product': 'whatsapp', 'contacts': [{'input': '12033002267', 'wa_id': '12033002267'}], 'messages': [{'id': 'wamid.HBgLMTIwMzMwMDIyNjcVAgARGBIzNTNFNTlFQ0I5RUQzNUIxRDgA', 'message_status': 'accepted'}]}


In [ ]:
# Seguir conversando con el bot para interactuar con Llama

payload_2 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": "12033002267", "text": {"body": "¿A qué hora abren los sábados?"}}
    ]}}]}]
}

r2 = requests.post("http://127.0.0.1:8000/webhook", json=payload_2)
print("Segundo mensaje:", r2.status_code, r2.text)

# la respuesta demora aprox. de 1 a 2 min. al ejecutar local en colab

INFO:     127.0.0.1:55838 - "POST /webhook HTTP/1.1" 200 OK
Segundo mensaje: 200 {"status":"ok"}


## **CHALLENGE: ASISTENTE DE PEDIDOS**

Una vez visto el ***Hands-On: Hands-On: WhatsApp Cloud API***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

A partir del asistente de pedidos construido, se agregan dos mejoras:

* Un mensaje de bienvenida para usuarios nuevos.
* Un tono de respuesta más controlado.

**IMPORTANTE:** Para su revisión, es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.

### **INSTRUCCIONES:**

**1. Prepara el entorno:** Instala y arranca Ollama en Colab (no viene preinstalado en la máquina virtual, así que se instala y se arranca en segundo plano al inicio de la sesión), descarga el modelo Llama 3.1 8B, y carga las credenciales de WhatsApp desde **Colab Secrets**.

In [ ]:
# Instalar Ollama


In [ ]:
# Iniciar el servidor de Ollama


In [ ]:
# Leer credenciales desde Colab Secrets


**2. Configura el servidor y el registro de usuarios conocidos:**

   * Crea el servidor FastAPI con las estructuras en memoria necesarias: `mensajes_recibidos` para la bitácora de payloads, y `usuarios_conocidos` (un `set`) para llevar registro de quién ya recibió el saludo.
   * Define la función `normalizar_numero_mx` para que los números mexicanos se envíen en el formato correcto.

In [ ]:
# Configurar el servidor del webhook


In [ ]:
# Definir función de normalización de número


**3. Genera la respuesta con un tono controlado:** Ajusta el prompt que se envía a Llama para que las respuestas se mantengan breves (máximo 2-3 líneas) y con un tono amable.

In [ ]:
# Definir función de generación de respuesta


**4. Define los endpoints del webhook:**

   * Crea el endpoint de verificación (`GET /webhook`).
   * Crea el endpoint de recepción de mensajes (`POST /webhook`), delegando el trabajo pesado a `BackgroundTasks` para responder de inmediato a Meta sin esperar a Ollama.

In [ ]:
# Definir endpoint de verificación de credenciales


In [ ]:
# Definir endpoint de recepción de mensajes


**5. Agrega el saludo inicial y el envío de la respuesta:**

   * Define la función que envía el mensaje de WhatsApp.
   * Define la función que decide si se manda el saludo fijo o la respuesta generada por Llama, según si el número ya está en `usuarios_conocidos`.

In [ ]:
# Definir función de envío de respuesta


In [ ]:
# Definir función de procesamiento de respuesta


**6. Expón el servidor y prueba el flujo completo:**

   * Instala `pyngrok`, abre el túnel público y arranca el servidor en un hilo aparte.
   * Manda un primer mensaje de prueba y confirma que se recibe el saludo; manda un segundo mensaje y confirma que ya llega la respuesta generada por Llama.

In [ ]:
# Installar e iniciar tunel publico con ngrok


In [ ]:
# Mandar mensaje al bot para recibir un saludo


In [ ]:
# Seguir conversando con el bot para interactuar con Llama
